## Leitura dos dados brutos, seleção de colunas, tratamento de nulos

### Importações e Configurações

In [1]:
import os
import glob
import pandas as pd
import numpy as np

# Configurações de exibição do pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


### Definição dos Caminhos e Colunas de Interesse

In [2]:
# Diretórios do projeto
RAW_DATA_PATH = '../data/raw/'
PROCESSED_DATA_PATH = '../data/processed/'

# Colunas selecionadas de acordo com o plano de trabalho
COLUNAS_INTERESSE = [
    'id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 
    'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 
    'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 
    'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 
    'feridos_graves', 'ilesos', 'latitude', 'longitude'
]

### Função para Carregamento e Concatenação dos CSVs da PRF

In [3]:
def carregar_dados_prf(caminho_pasta, colunas):
    """
    Carrega todos os arquivos CSV presentes na pasta de dados brutos.
    Os dados abertos da PRF utilizam codificação latin1 e separador ';'.
    """
    arquivos_csv = glob.glob(os.path.join(caminho_pasta, "*.csv"))
    print(f"Arquivos CSV encontrados: {arquivos_csv}")
    
    lista_dfs = []
    
    for arquivo in arquivos_csv:
        print(f"Lendo: {arquivo}...")
        df_temp = pd.read_csv(
            arquivo, 
            sep=';', 
            encoding='latin1', 
            low_memory=False
        )
        
        # Padroniza nomes de colunas em minúsculas
        df_temp.columns = [c.strip().lower() for c in df_temp.columns]
        
        # Filtra apenas as colunas que constam no arquivo e estão na lista
        cols_existentes = [col for col in colunas if col in df_temp.columns]
        df_temp = df_temp[cols_existentes]
        
        lista_dfs.append(df_temp)
        
    df_unificado = pd.concat(lista_dfs, ignore_index=True)
    print(f"\nBase consolidada: {df_unificado.shape[0]} registros e {df_unificado.shape[1]} colunas.")
    return df_unificado

df_bruto = carregar_dados_prf(RAW_DATA_PATH, COLUNAS_INTERESSE)

Arquivos CSV encontrados: ['../data/raw\\datatran2023.csv', '../data/raw\\datatran2024.csv', '../data/raw\\datatran2025.csv']
Lendo: ../data/raw\datatran2023.csv...
Lendo: ../data/raw\datatran2024.csv...
Lendo: ../data/raw\datatran2025.csv...

Base consolidada: 213451 registros e 24 colunas.


### Tratamento de Nulos, Tipos e Inconsistências

In [4]:
df = df_bruto.copy()

# 1. Tratamento de Datas e Criação de Features Temporais
df['data_inversa'] = pd.to_datetime(df['data_inversa'], errors='coerce')
df['ano'] = df['data_inversa'].dt.year
df['mes'] = df['data_inversa'].dt.month

# Extração da hora numérica (0 a 23)
df['hora'] = pd.to_datetime(df['horario'], format='%H:%M:%S', errors='coerce').dt.hour

# 2. Tratamento das Coordenadas Geográficas (vírgula para ponto e float)
for coord in ['latitude', 'longitude', 'km']:
    if coord in df.columns:
        df[coord] = df[coord].astype(str).str.replace(',', '.')
        df[coord] = pd.to_numeric(df[coord], errors='coerce')

# 3. Tratamento de Valores Numéricos de Vítimas
cols_vitimas = ['mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'pessoas']
for col in cols_vitimas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# 4. Padronização de Strings e Categóricas
cols_categoricas = [
    'classificacao_acidente', 'fase_dia', 'condicao_metereologica', 
    'tipo_pista', 'tracado_via', 'uso_solo', 'dia_semana'
]
for col in cols_categoricas:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        # Converte strings de nulos comuns
        df[col] = df[col].replace({'NONE': np.nan, 'NULL': np.nan, 'IGNORADO': np.nan, 'NÃO INFORMADO': np.nan})

print("Tipos de dados convertidos e padronizados com sucesso.")

Tipos de dados convertidos e padronizados com sucesso.


### Limpeza Espacial de Coordenadas (Filtro Brasil)

In [5]:
# Filtro de limites geográficos coerentes para o território brasileiro
# Latitude: ~ -34° a +5.5° | Longitude: ~ -74° a -34°
linhas_iniciais = len(df)

coord_validas = (
    (df['latitude'].between(-34.0, 5.5)) & 
    (df['longitude'].between(-74.0, -34.0))
)

df = df[coord_validas].copy()
print(f"Registros removidos por coordenadas inválidas/fora do Brasil: {linhas_iniciais - len(df)}")
print(f"Total restante: {len(df)}")

Registros removidos por coordenadas inválidas/fora do Brasil: 1
Total restante: 213450


### Engenharia de Variáveis e Alvo (Severidade e Índice de Severidade)

In [6]:
# 1. Alvo Categórico para o Random Forest:
# Mapear para classes estruturadas (0: Sem Vítimas, 1: Com Vítimas Feridas, 2: Com Vítimas Fatais)
mapa_severidade = {
    'SEM VÍTIMAS': 'SEM_VITIMAS',
    'COM VÍTIMAS FERIDAS': 'COM_FERIDOS',
    'COM VÍTIMAS FATAIS': 'FATAL'
}
df['alvo_severidade'] = df['classificacao_acidente'].map(mapa_severidade)

# 2. Índice de Severidade Ponderado (UPS/IPEA - Pesos: 1 sem vítimas, 5 com feridos, 13 com óbitos)
# Referência: feridos = leves + graves
df['indice_severidade'] = (
    (df['ilesos'] * 0) + 
    (df['feridos_leves'] * 1) + 
    (df['feridos_graves'] * 5) + 
    (df['mortos'] * 13)
)

print("Distribuição da Variável Alvo Categórica:")
print(df['alvo_severidade'].value_counts(dropna=False))

Distribuição da Variável Alvo Categórica:
alvo_severidade
COM_FERIDOS    164278
SEM_VITIMAS     33880
FATAL           15289
NaN                 3
Name: count, dtype: int64


### Inspeção Final e Exportação do Dataset Processado

In [7]:
# Criar diretório processado se não existir
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Remoção de sinistros sem classificação do alvo
df_final = df.dropna(subset=['alvo_severidade']).reset_index(drop=True)

# Exportação do dataset limpo para a pasta processed
saida_csv = os.path.join(PROCESSED_DATA_PATH, 'prf_2023_2025_limpo.csv')
df_final.to_csv(saida_csv, index=False, sep=';', encoding='utf-8')

print(f"Processamento concluído com sucesso!")
print(f"Arquivo salvo em: {saida_csv}")
print(f"Dimensões finais: {df_final.shape}")
df_final.head()

Processamento concluído com sucesso!
Arquivo salvo em: ../data/processed/prf_2023_2025_limpo.csv
Dimensões finais: (213447, 29)


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,latitude,longitude,ano,mes,hora,alvo_severidade,indice_severidade
0,496519.0,2023-01-01,DOMINGO,02:00:00,ES,101,114.0,SOORETAMA,Ausência de reação do condutor,Saída de leito carroçável,COM VÍTIMAS FERIDAS,PLENA NOITE,Crescente,CÉU CLARO,SIMPLES,RETA,NÃO,1,0,1,0,0,-19.094849,-40.050958,2023,1,2,COM_FERIDOS,1
1,496543.0,2023-01-01,DOMINGO,03:40:00,SP,116,113.1,TAUBATE,Entrada inopinada do pedestre,Atropelamento de Pedestre,COM VÍTIMAS FATAIS,PLENA NOITE,Decrescente,CÉU CLARO,DUPLA,RETA,SIM,5,1,0,0,0,-23.044566,-45.582598,2023,1,3,FATAL,13
2,496610.0,2023-01-01,DOMINGO,10:40:00,PR,376,314.8,ORTIGUEIRA,Velocidade Incompatível,Tombamento,SEM VÍTIMAS,PLENO DIA,Crescente,SOL,DUPLA,CURVA,NÃO,2,0,0,0,1,-23.985512,-51.083555,2023,1,10,SEM_VITIMAS,0
3,496659.0,2023-01-01,DOMINGO,14:55:00,MG,116,569.4,MANHUACU,Acumulo de água sobre o pavimento,Colisão frontal,COM VÍTIMAS FERIDAS,PLENO DIA,Decrescente,CHUVA,SIMPLES,DECLIVE;CURVA,NÃO,4,0,0,2,1,-20.100075,-42.178841,2023,1,14,COM_FERIDOS,10
4,496671.0,2023-01-01,DOMINGO,15:45:00,MG,262,569.8,CORREGO DANTA,Condutor Dormindo,Saída de leito carroçável,SEM VÍTIMAS,PLENO DIA,Decrescente,NUBLADO,SIMPLES,RETA;ACLIVE,SIM,2,0,0,0,1,-19.716043,-46.021922,2023,1,15,SEM_VITIMAS,0
